# Module 4.4: Isolation Forests for Early Machine Breakdown Warning
### 🇵🇱 Lasy Izolacyjne (Isolation Forest) do Wykrywania Uszkodzeń

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/04_04_isolation_forest.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Train tree-based isolation ensembles to assign continuous anomaly scores to impending mechanical seizures.  
> 🇵🇱 **Streszczenie:** Wytrenuj las izolacyjny do wyliczania ciągłego wskaźnika anomalii zwiastującego zatarcie mechanizmu.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Operational Regimes**: Multi-mode CNC machining (Standby, Roughing, High-Speed Finishing) exhibit distinct physical baselines.
2. **K-Means Centroids**: Act as golden reference signatures for healthy machine cycles.
3. **DBSCAN & Isolation Forests**: Detect zero-shot anomalies without requiring historical breakdown labels.


In [ ]:
# 1. Simulate CNC Spindle Operational Regimes (RPM vs Motor Current)
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

np.random.seed(42)
# Regime 1: Idle (RPM: ~300, Current: ~6A)
c1 = np.random.multivariate_normal([300, 6], [[400, 5], [5, 0.8]], size=150)
# Regime 2: Normal Milling (RPM: ~2800, Current: ~32A)
c2 = np.random.multivariate_normal([2800, 32], [[2500, 20], [20, 4.0]], size=200)
# Regime 3: Heavy Roughing (RPM: ~2200, Current: ~65A)
c3 = np.random.multivariate_normal([2200, 65], [[2000, 30], [30, 8.0]], size=120)

# Unmodeled Breakdown Anomalies (e.g. tool seizure or mechanical bind)
anomalies = np.array([
    [1200, 58], [1500, 72], [3200, 80], [400, 35], [2600, 12]
])

X = np.vstack([c1, c2, c3, anomalies])
print(f"Total operational points: {len(X)}")


In [ ]:
# 2. Fit K-Means Regimes and Isolation Forest Anomaly Detection
kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
iso_forest = IsolationForest(contamination=0.03, random_state=42).fit(X)
anomaly_pred = iso_forest.predict(X)  # -1 represents anomaly

plt.figure(figsize=(11, 6))
# Plot normal clusters
plt.scatter(X[anomaly_pred == 1, 0], X[anomaly_pred == 1, 1], c=kmeans.labels_[anomaly_pred == 1], cmap='viridis', alpha=0.7, edgecolors='none', label='Healthy Regimes')
# Plot flagged anomalies
plt.scatter(X[anomaly_pred == -1, 0], X[anomaly_pred == -1, 1], color='#ef4444', s=120, edgecolors='black', linewidth=1.5, marker='o', label='Isolated Machine Anomalies')
# Plot cluster centroids
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=200, linewidth=2, label='Regime Centroids')

plt.title("Unsupervised Machine Regime Discovery & Anomaly Envelopes", fontsize=13, fontweight='bold')
plt.xlabel("Spindle Speed [RPM]")
plt.ylabel("Motor Current [Amps]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Instantiate sklearn.ensemble.IsolationForest on healthy baseline runs *(🇵🇱 Zainicjuj IsolationForest na danych ze sprawnego cyklu produkcyjnego)*
- [ ] **Task 2:** Compute decision_function continuous anomaly scores *(🇵🇱 Wylicz ciągły wskaźnik anomalii za pomocą decision_function)*
- [ ] **Task 3:** Set plant alert trigger threshold at anomaly score > 0.80 *(🇵🇱 Ustaw próg powiadomień alarmowych na poziomie wskaźnika > 0.80)*


In [ ]:
# Exercise Challenge:
# Verify that the known synthetic anomalies were successfully flagged by Isolation Forest
flagged_count = (anomaly_pred == -1).sum()
print(f"Flagged Anomalies: {flagged_count} (Expected 5 to 15)")
assert flagged_count >= 5, "Anomaly detector failed to isolate unmodeled states!"
print("✓ Unsupervised clustering exercise passed.")
